In [331]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pprint import pprint

from session_extraction import agenda_items, speech

ImportError: cannot import name 'agenda_items' from 'session_extraction' (/Users/susanne/Desktop/Lennart's Ordner/Data_Engineering_Projects/Bundestag_New/session_extraction.py)

## Fetching & Exploring an Bundestag plenary Seesion xml file
To start exploring we will try to fetch an xml file, and navigate it as a tree.
Once we are comfortable navigating the file we'll try flattening it into a flat table
/ a flat pandas df as the first step of denormilization (flattening a nested file structure),
so that we can after further normalize into different tables.

In [ ]:
file_path ="https://www.bundestag.de/resource/blob/1140642/21057.xml"

response = requests.get(file_path)
response.raise_for_status()
print(response.status_code)
print(type(response))


In [ ]:
# after fetching the xml using an https request
# we can access it as an tree by  converting it to
# an elementtree object, which is easier to navigate/work with
# for that we use pythons built in xml ElementTree object
# which represents an xml file, as the name suggests, as a tree
# of elements. We imported this type of object already so we can
# can make use of that class to instantiate an instance that holds
# the information of our plenary protocoll.

# For that we use the fromstring method, which expects, as the name suggests, as string
# as input. To access the our html response to a string, we can access its content attribute.

# Each tree has one core node, also referred to as the trunk of a tree.
# In this case we refer to it as the root of our element tree.
# We now pass our xml as a string, and get an elementree that we assign
# to root
# we see that we succesfully get an Elementree element (as the root is always returned, which holds
# all subelements/children of the rest of the tree), remember we're working with a hirarchical # datastructure here, we can access the root elements name by accessing its tag attribute,
# as well as see that the root holds 4 subelements which are its children
root = ET.fromstring(response.content)
print(type(root))
print(root.tag)
print(len(root))

In [ ]:
# in addition to that an element is of a certain type (its name), can hold on to values, usually text
# as well as can have attributes
# In our case our root element is called "dbtplenarprotokoll" which translates to "dbtplenaryprotocol"
# but holds varius attributes as metadata
# those attributes are returned as a python dictionary, which means that we can access
# them using their keys
# We will make use of that to get our plenary level metadata like session date, legislative
# period, session number etc
print(root.attrib)
print(type(root.attrib))
plenary_session_metadata = root.attrib

In [ ]:
# to test we try to access the legaslative period, session nr and
# date, later when flattening the file we can make use of this to
# add session level metadata to each speech
legaslative_period = plenary_session_metadata["wahlperiode"]
session_nr = plenary_session_metadata["sitzung-nr"]
session_date = plenary_session_metadata["sitzung-datum"]
print(f"Legaslative Period: {legaslative_period}")
print(f"Session Nr: {session_nr}")
print(f"Session Date: {session_date}")

In [ ]:
# ok now that we've covered everyhting of the root node
# we can move down the tree and explore its sub elements, also
# referred to as its child
for child in root:
    print(child.tag)

# in our case the root holds 4 children:
# the vorspann, the actual undergoings of the session in
# sitzungsverlauf which also hold the individual speeches,
# additional information in anlagen
# and lastly a speakerlist, which later might be useful to validate
# our extracted speeches against
# Of most interest to us is the sitzungsverlauf which contains
# the core of our speeches


In [ ]:
# Among other ways the two most straightforward ways of accessing sub-elements/children of an xml element
# are accessing them by their location (index) or by their tag.
# The index is used like with any other iterable (like a list), to get elements
# of a certain tag we can use the find or findall methods
print(f"Index: {root[0].tag}\nTag: {root.find("vorspann").tag}")


In [ ]:
# for convenience we will get hold of the 4 main sections
# of the plenary protocol
opening = root.find("vorspann")
agenda = root.find("sitzungsverlauf")
attachment = root.find("anlagen")
speaker_list = root.find("rednerliste")

# lets check
print(f"Opening: {opening.tag}")
print(f"Agenda: {agenda.tag}")
print(f"Attachment: {attachment.tag}")
print(f"Opening: {speaker_list.tag}")


In [ ]:
# As the core of the session of our interest is the main part
# as it holds all our speeches, lets dig deeper and check which subelements
# it consists of
print(f"Nr. of main elements: {len(agenda)}\n")

print("Main elements (Name and attributes): \n")
for child in agenda:
     print(child.tag)
     print(child.attrib)

# What wenotice is that the agenda containts and opening and closing,
# as well as agenda items which represent debates

In [ ]:
# lets get hold of the agenda items as a list
# and explore the tags, attributes and nr. of subelements of the
# first agenda item
agenda_items = agenda.findall("tagesordnungspunkt")
first_agenda_item = agenda_items[0]
print(f"Agenda item name: {first_agenda_item.tag}")
print(f"Agenda attributes: {first_agenda_item.attrib}")
print(f"Nr. of sub elements: {len(first_agenda_item)}")

# We find the tag name, as expected, the agenda items title as the top-id attribute
# and 33 sub-elements, which the majority of will be the speeches of that debate

In [ ]:
# lets check what sub elements
# the agenda item holds
for item in first_agenda_item:
    print(item.tag)

# We find a lot of p tags (paragraphs), many "rede" tags (translates to speech)
# and few "kommentar" tags (translates to comment). Manual investigations showed that first
# paragraphs often  serve as a debate opening that entails the title, sub elements fo the debate
# and sometimes documents that the debate refers to.
# This is followed by speeches and closing paragraphs. The comment tags seem to be the exception rather
# than the norm, but need to be further investigated

In [332]:
# Lets keep our focus with the first agenda item and dig deeper
# For this lets get all paragraphs and all speeches
paragraphs = first_agenda_item.findall("p")
speeches = first_agenda_item.findall("rede")
nr_speeches = len(speeches)
print(f"Nr. of paragraphs: {len(paragraphs)}")
print(f"Nr. of speeches: {nr_speeches}")

Nr. of paragraphs: 14
Nr. of speeches: 15


In [333]:
# It turns out that most of the paragraphs are actually leaves of our element tree,
# meaning that they have no sub-elements/children and thus present the last nodes of that particular branch.
# Lets explore them and their attributes which also surfaces that we need another method
# to target specific paragraphs and lastly get their values
for p in paragraphs:
    print(f"Paragraph Attributes: {p.attrib}")
    print(f"Paragraph values: {p.text}")

# We see that all pargraphs hold a "klasse" attributes (translates to class)
# All p's with class values of "T_fett" hold the title of the agenda item
# The ones with "T_NaS" seem to hold subtitles
# If the agenda relates to documents, we find a p of class "T_Drs"
#  that hold an sub anchor elements with an urlpath that links to that document

Paragraph Attributes: {'klasse': 'J'}
Paragraph values: Ich rufe die Tagesordnungspunkte 7a und 7b auf: 
Paragraph Attributes: {'klasse': 'T_NaS'}
Paragraph values: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Paragraph Attributes: {'klasse': 'T_fett'}
Paragraph values: 			Zum Jahreswirtschaftsbericht 2026
Paragraph Attributes: {'klasse': 'T_NaS'}
Paragraph values: 		b)	Beratung der Unterrichtung durch die Bundesregierung 
Paragraph Attributes: {'klasse': 'T_fett'}
Paragraph values: 			Jahreswirtschaftsbericht 2026 der Bundesregierung
Paragraph Attributes: {'klasse': 'T_Drs'}
Paragraph values: 			Drucksache 
Paragraph Attributes: {'klasse': 'T_Ueberweisung'}
Paragraph values: 			Überweisungsvorschlag: Ausschuss für Wirtschaft und Energie (f) Ausschuss für Recht und Verbraucherschutz Finanzausschuss Ausschuss für Landwirtschaft, Ernährung und Heimat Ausschuss für Arbeit und Soziales Ausschuss für Bildung, Familie, Senioren, Frauen und Ju

In [334]:
# Lets get hold on the titles and document url
main_title = first_agenda_item.findall("p[@klasse='T_fett']")
for title in main_title:
    print(title.text)

			Zum Jahreswirtschaftsbericht 2026
			Jahreswirtschaftsbericht 2026 der Bundesregierung


In [335]:
# lets get hold of sub-elements
# ok lets explore a new method/concept that helps us to
# identify specific elements of our interest, by not only
# defining the tag to look for but also the attribute and if
# needed also the hierarchy level. We can still use find or findall
# but this time we not only the tagname but an xpath.
# When looking for a a p element in the children of the
# first agenda item with the attribute "klasse": "T_NaS"
sub_titles = first_agenda_item.findall("p[@klasse = 'T_fett']")
for title in sub_titles:
    print(title.text)
# In our case we looked for the title(s) of the first agenda item
# which is about the yearly economic report of 2026

			Zum Jahreswirtschaftsbericht 2026
			Jahreswirtschaftsbericht 2026 der Bundesregierung


In [336]:
# Important to know: findall returns an empty list
# if no element matches the defined mask/filter
ok = first_agenda_item.findall("p[@klasse= 'lennart']")
print(ok)

[]


In [337]:
# We can also further define multiple hierarchy levels to navigate
# through by adding backslashes how we would with a filepath too
# We make use of that to look for the document element that
# the first agenda item refers to.
# Further we find a new method that lets us access attributes of an
# element using their attribute name as a key, the get method.
# It lets us specify which attribute to look for and which value
# to return when that attribute is not found
document = first_agenda_item.findall("p[@klasse = 'T_Drs']/a")
print(document_url.get("href", None))
# This time we get hold of the documents url that the first agenda item relates
# too. We saw earlier that the first agenda item is dedicated towards the
# yearly economic report of 2026, the link we just accessed lead to that
# exact report.

https://dserver.bundestag.de/btd/21/037/2103700.pdf


In [338]:
# For testing purposes lets loop over all agenda items
# and try to get the respective doc urls they relate to.
for item in agenda_items:
    documents = item.findall("p[@klasse = 'T_Drs']/a")
    for doc in documents:
        print(doc.get("href", "No URL"))

https://dserver.bundestag.de/btd/21/037/2103700.pdf
https://dserver.bundestag.de/btd/21/036/2103661.pdf
https://dserver.bundestag.de/btd/21/038/2103842.pdf
https://dserver.bundestag.de/btd/21/038/2103843.pdf
https://dserver.bundestag.de/btd/21/036/2103619.pdf
https://dserver.bundestag.de/btd/21/028/2102804.pdf


In [339]:
# Ok the wrap our paragraph exploration, lets get the title, sub title(s) and document links,
# test for all agenda items, so that we can move on to exploring speeches.
# NOTE: We will for now ignore agenda item openings made by the president, which usually is a short introduction
#       of what is to be debated etc.

titles = first_agenda_item.findall("p[@klasse= 'T_fett']")
sub_titles = first_agenda_item.findall(("p[@klasse = 'T_NaS']"))
documents = first_agenda_item.findall("p[@klasse='T_Drs']/a")

print("Title(s):")
for title in titles:
    print(f"Agenda Item Titel: {title.text}")

print("\nSubtitle(s):")
for subtitle in sub_titles:
    print(f"Agenda Item Titel: {subtitle.text}")

print("\nDocument link(s):")

for doc in documents:
    print(f"Document link: {doc.get('href', 'No link found')}")

Title(s):
Agenda Item Titel: 			Zum Jahreswirtschaftsbericht 2026
Agenda Item Titel: 			Jahreswirtschaftsbericht 2026 der Bundesregierung

Subtitle(s):
Agenda Item Titel: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Agenda Item Titel: 		b)	Beratung der Unterrichtung durch die Bundesregierung 

Document link(s):
Document link: https://dserver.bundestag.de/btd/21/037/2103700.pdf


In [340]:
# For testing purposes let extend this to all agenda items
for item in agenda_items:
    print(50*"-")
    print(f"Agenda item: {item.tag}")

    titles = item.findall("p[@klasse= 'T_fett']")
    sub_titles = item.findall(("p[@klasse = 'T_NaS']"))
    documents = item.findall("p[@klasse='T_Drs']/a")

    print("\nTitle(s):")
    for title in titles:
        print(f"Agenda Item Titel: {title.text}")

    print("\nSubtitle(s):")
    for subtitle in sub_titles:
        print(f"Agenda Item Titel: {subtitle.text}")

    print("\nDocument link(s):")

    for doc in documents:
        print(f"Document link: {doc.get('href', 'No link found')}")

--------------------------------------------------
Agenda item: tagesordnungspunkt

Title(s):
Agenda Item Titel: 			Zum Jahreswirtschaftsbericht 2026
Agenda Item Titel: 			Jahreswirtschaftsbericht 2026 der Bundesregierung

Subtitle(s):
Agenda Item Titel: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Agenda Item Titel: 		b)	Beratung der Unterrichtung durch die Bundesregierung 

Document link(s):
Document link: https://dserver.bundestag.de/btd/21/037/2103700.pdf
--------------------------------------------------
Agenda item: tagesordnungspunkt

Title(s):
Agenda Item Titel: 		Mobilitätsgarantie einführen – Produktionskapazitäten für die Verkehrswende aufbauen
Agenda Item Titel: 		Gemeindeverkehrsfinanzierungsgesetz novellieren – Kommunen stärken und Ausbau des öffentlichen Personennahverkehrs langfristig absichern

Subtitle(s):
Agenda Item Titel: 	24	Beratung des Antrags der Abgeordneten Luigi Pantisano, Marcel Bauer, Lorenz Gösta Beutin, w

In [341]:
# Ok now that we've covered the most fundamental attributes of an agenda item's
# pragraph elements, namely the titles, subtitles and documents they relate too.
# NOTE: We focus on the fundamentals here, edge cases and exceptions will be dealt with
#       once we've got an MVP
# We now move on to explore the structure of speeches. This will be sub divided into
# speaker level information like name, party affiliation and role and speech level
# information which deals with the speeches actual content as well as interruptions.

# As always lets break things down to a single unit for initial exploration,
# in this case a single speech (we pick a random one)
speech = speeches[9]

print(f"Speech tag: {speech.tag}")
print(f"Nr of elements in a speech: {len(speech)}")
type_of_elements = {}

for child in speech:
    if not child.tag in type_of_elements:
        type_of_elements[child.tag] = {"count": 1,
                                       "attributes": set(),
                                       "attribute_values": set()
                                       }
        element_tacker = type_of_elements[child.tag]
        for key, value in child.attrib.items():

            element_tacker["attributes"].add(key)
            element_tacker["attribute_values"].add(value)

    else:
        element_tracker = type_of_elements[child.tag]
        element_tracker["count"] +=1
        if child.attrib:
            for key, value in child.attrib.items():
                element_tacker["attributes"].add(key)
                element_tacker["attribute_values"].add(value)

print(f"Element tracker:")
pprint(type_of_elements)
# As part of our discovery I came up with the idea to build a schema discovery tool,
# to map out and track schema's to come up with a way to identify structures across branches and sessions
# that is more standardized and scalable than eyballing sample elements
# This marks a very first simple version of it, as my main focus for now is to build an MVP
# of the overall pipeline focusing on standard cases, this will for now not be extended for
# now. But for professionalizing, standardizing and scaling the pipeline this will
# definitely build out!

Speech tag: rede
Nr of elements in a speech: 39
Element tracker:
{'kommentar': {'attribute_values': {'J', 'O'},
               'attributes': {'klasse'},
               'count': 15},
 'name': {'attribute_values': {'redner', 'J_1', 'J'},
          'attributes': {'klasse'},
          'count': 2},
 'p': {'attribute_values': {'redner', 'J_1'},
       'attributes': {'klasse'},
       'count': 22}}


In [342]:
# Simply as a quick test, lets extend the element tracker
# to all speeches
type_of_elements = {}

for speech in speeches:
    for child in speech:
        if not child.tag in type_of_elements:
            type_of_elements[child.tag] = {"count": 1,
                                           "attributes": set(),
                                           "attribute_values": set()
                                           }
            element_tacker = type_of_elements[child.tag]
            for key, value in child.attrib.items():

                element_tacker["attributes"].add(key)
                element_tacker["attribute_values"].add(value)

        else:
            element_tracker = type_of_elements[child.tag]
            element_tracker["count"] +=1
            if child.attrib:
                for key, value in child.attrib.items():
                    element_tacker["attributes"].add(key)
                    element_tacker["attribute_values"].add(value)

print(f"Element tracker:")
pprint(type_of_elements)

Element tracker:
{'kommentar': {'attribute_values': {'J', 'O'},
               'attributes': {'klasse'},
               'count': 171},
 'name': {'attribute_values': {'redner', 'J_1', 'J', 'O'},
          'attributes': {'klasse'},
          'count': 26},
 'p': {'attribute_values': {'redner', 'J_1', 'J'},
       'attributes': {'klasse'},
       'count': 321}}


In [343]:
# Ok lets move to explore the speaker info
# Speeches elements actually contain an element that is specifically
# dedicated to speaker metadata, inlcuding a speaeker id as an attribute,
# first name, last name, party afilation or role if part of government as sub-elements.
# These are all elements we aim to extract.
# Lets also collect speaker id which we can collect in a set and
# in the end compare against the official speaker list, to validate that
# the speakers we found speeches for matches the speaker list provided


speaker_id = speech.find(".//redner").get("id", None)
name = speech.find(".//vorname")
lastname = speech.find(".//nachname")
role = speech.find(".//rolle_lang")
party_affiliation = speech.find(".//fraktion")


In [344]:
print(f"Speaker ID: {speaker_id}")
print(f"Firstname: {name.text}")
print(f"Lastname: {lastname.text}")
if party_affiliation is not None:
    print(f"Party:: {party_affiliation.text}")
if role is not None:
    print(f"Role: {role.text}")

# Appears to work for the individual speech, lets test
# if it also works for the other speeches of this session

Speaker ID: 11005068
Firstname: Fabian
Lastname: Gramling
Party:: CDU/CSU


In [345]:
# Lets test that logic on all speeches
speakers_extracted = set()

for item in agenda_items:
    speeches = item.findall("rede")

    for speech in speeches:
        speaker_id = speech.find(".//redner").get("id", None)
        name = speech.find(".//vorname")
        lastname = speech.find(".//nachname")
        role = speech.find(".//rolle_lang")
        party_affiliation = speech.find(".//fraktion")
        speakers_extracted.add(speaker_id)

        print(f"Speaker ID: {speaker_id}")
        print(f"Firstname: {name.text}")
        print(f"Lastname: {lastname.text}")
        if party_affiliation is not None:
            print(f"Party:: {party_affiliation.text}")
        if role is not None:
            print(f"Role: {role.text}")
        print(50* "+")

# Our logic seems to hold across all speeches, a speaker can either be afiliated with
# a party or hold a role such a minister, president, chancelor etc.

Speaker ID: 11003209
Firstname: Katherina
Lastname: Reiche
Role: Bundesministerin für Wirtschaft und Energie
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11004761
Firstname: Leif-Erik
Lastname: Holm
Party:: AfD
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11005267
Firstname: Armand
Lastname: Zorn
Party:: SPD
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11005016
Firstname: Felix
Lastname: Banaszak
Party:: BÜNDNIS 90/DIE GRÜNEN
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11004832
Firstname: Sepp
Lastname: Müller
Party:: CDU/CSU
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11005260
Firstname: Janine
Lastname: Wissler
Party:: Die Linke
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11005191
Firstname: Sebastian
Lastname: Roloff
Party:: SPD
++++++++++++++++++++++++++++++++++++++++++++++++++
Speaker ID: 11004787
Firstname: Enrico
Lastname: Komning
Party:: AfD
+++++++++++++++++++++++++++

In [346]:
# lets add all speakers id's of speeches we find to a set
# do the same for the officially provided speaker list and compare the two
# to check if we found speeches of all speaker we were supposed to find

print(len(speakers_extracted))
print(speakers_extracted)

speakers = speaker_list.findall("redner")
speakers_listed = {speaker.get("id",None) for speaker in speakers}
print(len(speakers_listed))
print(speakers_listed)


72
{'11003544', '11004905', '11005494', '11005491', '11005477', '11003772', '11005593', '11004773', '11005402', '11005563', '11005588', '11005260', '11004038', '11004761', '11005067', '11005063', '11004933', '11005143', '11005583', '11005408', '11005478', '11005118', '11004440', '11004792', '11004672', '11005106', '11005618', '11005068', '11004262', '11004396', '11005077', '11005613', '11005190', '11004437', '11005499', '11005103', '11005019', '11004724', '11005458', '11005016', '11005267', '11005115', '11005539', '11005036', '11005553', '11005432', '11005039', '11005483', '11005065', '11005257', '11004669', '11003034', '11004381', '11005448', '11005191', '11005410', '11004832', '11004659', '11003209', '11003638', '11005623', '11005419', '11004050', '11005476', '11005102', '11005134', '11004752', '11005137', '11003218', '11004787', '11004893', '11005240'}
72
{'11003544', '11004905', '11005494', '11005491', '11005477', '11003772', '11005593', '11004773', '11005402', '11005563', '1100558

In [347]:
speakers_not_extracted = []

for speaker_id in speakers_extracted:
    if not speaker_id in speakers_listed:
        speakers_not_extracted.append(speaker_id)
        print(speaker_id)

print(f"Nr. of speakers in speaker list but for which we didn't find speeches: {len(speakers_not_extracted)}")
if speakers_not_extracted:
    print(F"ID's of those speakers: {speakers_not_extracted}")

# It seems like we found speeches for all speakers provided in the official
# speaker list if we did extract all speakers from the official speaker list
# correctly

Nr. of speakers in speaker list but for which we didn't find speeches: 0


In [348]:
# Lets now move on to explore the actual speeches
# NOTE: Speeches also contain interruptions, like a member of another party
# comments on a speech, or a party reacts during a speech e.g. by applauding
# for simplicity we will first extract all speech content as one string, including the interruptions
# /comments. Later we will extend our logic to extract interruptions/comments and store them together
# with their location within the speech, so that it will be possible to reconstruct speeches and their
# interupptions
# As always lets start with one single element, a speech, and slowly expand.


accepted_class_attributes = ("J_1", "J", "O")

speech_paragraphs = speech.findall("p")

speech_chunks =[]
comments = []

position = 0
for element in speech:

    if element.tag == "p" and element.text:
        speech_chunks.append(element.text)
        position += (len(element.text) + 1)
    elif element.tag == "kommentar" and element.text:
        comments.append({"index_position": position,
                         "comment_text": element.text})

print(" ".join(speech_chunks))
print(comments)
print(60*"*")


Sehr geehrter Herr Präsident! Liebe Kolleginnen und Kollegen! Die Energiesicherheit, Energiesouveränität eines Landes lässt sich gut an den Maßnahmen ablesen, die ergriffen werden, wenn es ganz akut in einer Notsituation ist. Dann lässt sich die Beweisführung erbringen, was man aus der Situation heraus vernünftigerweise zu tun hat. Das zeigt etwa das Beispiel Ukraine. Die Hilfen, die dort zur Gewährleistung der Energiesicherheit über die Fondslösungen laufen, fokussieren sich auf dezentrale Systeme, auf erneuerbare Systeme, auf Bioenergie, auf Blockheizkraftwerke, auf kleine Lösungen. – Darf ich das Wort haben? In solchen Notsituationen lässt sich sehr gut erkennen, welche Lösungen wir brauchen, wenn es um Energiesicherheit geht. Und interessanterweise ist ja genau dies auch das Erfolgsrezept im Hochlauf der Energiewende über die letzten Jahrzehnte in Deutschland gewesen. Wir hatten zwar gewisse Rückschläge zu verzeichnen, weil Einschränkungen in den Rahmenbedingungen vorgenommen worde

In [349]:
print(speech_chunks)

['Sehr geehrter Herr Präsident! Liebe Kolleginnen und Kollegen! Die Energiesicherheit, Energiesouveränität eines Landes lässt sich gut an den Maßnahmen ablesen, die ergriffen werden, wenn es ganz akut in einer Notsituation ist. Dann lässt sich die Beweisführung erbringen, was man aus der Situation heraus vernünftigerweise zu tun hat. Das zeigt etwa das Beispiel Ukraine. Die Hilfen, die dort zur Gewährleistung der Energiesicherheit über die Fondslösungen laufen,', 'fokussieren sich auf dezentrale Systeme, auf erneuerbare Systeme,', 'auf Bioenergie, auf Blockheizkraftwerke, auf kleine Lösungen.', '– Darf ich das Wort haben?', 'In solchen Notsituationen lässt sich sehr gut erkennen, welche Lösungen wir brauchen, wenn es um Energiesicherheit geht. Und interessanterweise ist ja genau dies auch das Erfolgsrezept im Hochlauf der Energiewende über die letzten Jahrzehnte in Deutschland gewesen.', 'Wir hatten zwar gewisse Rückschläge zu verzeichnen, weil Einschränkungen in den Rahmenbedingungen 

In [350]:
print(comments)

[{'index_position': 464, 'comment_text': '(Stephan Brandner [AfD]: …, die wären bei uns auch gut aufgehoben!)'}, {'index_position': 530, 'comment_text': '(Karsten Hilse [AfD]: Ja, super! Stromgeneratoren aus Diesel sind jetzt plötzlich erneuerbar!)'}, {'index_position': 592, 'comment_text': '(Stephan Brandner [AfD]: „Kleine Lösungen“ heißt Lagerfeuer!\xa0– Manuel Krauthausen [AfD]: Das ist doch die Unwahrheit, was Sie hier verkünden!)'}, {'index_position': 619, 'comment_text': '(Stephan Brandner [AfD]: Haben Sie doch!\xa0– Karsten Hilse [AfD]: Selbstverständlich!)'}, {'index_position': 885, 'comment_text': '(Dr.\xa0Paul Schmidt [AfD]: Dieselgeneratoren?)'}, {'index_position': 1314, 'comment_text': '(Dr.\xa0Jan-Niclas Gesenhues [BÜNDNIS\xa090/DIE GRÜNEN]: Arbeitsplatzvernichtung!)'}, {'index_position': 1672, 'comment_text': '(Karsten Hilse [AfD]: Das sehen die Stromkunden anders, dass da keine Rechnung kommt!)'}, {'index_position': 2468, 'comment_text': '(Lachen des Abg. Dr.\xa0Ingo Hah

In [351]:
# Lets extend that to all speeches, to see if our logic holds

In [352]:
print(" ".join(speech_chunks))

Sehr geehrter Herr Präsident! Liebe Kolleginnen und Kollegen! Die Energiesicherheit, Energiesouveränität eines Landes lässt sich gut an den Maßnahmen ablesen, die ergriffen werden, wenn es ganz akut in einer Notsituation ist. Dann lässt sich die Beweisführung erbringen, was man aus der Situation heraus vernünftigerweise zu tun hat. Das zeigt etwa das Beispiel Ukraine. Die Hilfen, die dort zur Gewährleistung der Energiesicherheit über die Fondslösungen laufen, fokussieren sich auf dezentrale Systeme, auf erneuerbare Systeme, auf Bioenergie, auf Blockheizkraftwerke, auf kleine Lösungen. – Darf ich das Wort haben? In solchen Notsituationen lässt sich sehr gut erkennen, welche Lösungen wir brauchen, wenn es um Energiesicherheit geht. Und interessanterweise ist ja genau dies auch das Erfolgsrezept im Hochlauf der Energiewende über die letzten Jahrzehnte in Deutschland gewesen. Wir hatten zwar gewisse Rückschläge zu verzeichnen, weil Einschränkungen in den Rahmenbedingungen vorgenommen worde

In [353]:
speeches = first_agenda_item.findall("rede")

for speech in speeches:
    speech_paragraphs = speech.findall("p")

    speech_chunks =[]
    comments = []

    position = 1
    for element in speech:

        if element.tag == "p" and element.text:
            speech_chunks.append(element.text)
            position += (len(element.text) + 1)
        elif element.tag == "kommentar" and element.text:
            comments.append({"index_position": position,
                             "comment_text": element.text})

    print(20*"+","SPEECH:", 20*"+")
    print(" ".join(speech_chunks))

    print("\n")
    print(20*"+","COMMENTS:", 20*"+")
    print(comments)
    print(100*"*")
    print(2*"\n")

++++++++++++++++++++ SPEECH: ++++++++++++++++++++
Frau Präsidentin! Meine sehr geehrten Damen und Herren! Hinter uns liegen zwei Jahre Rezession – Rückwärtsgang. Hinter uns liegt ein Jahr Stagnation – Seitwärtsgang. Vor uns liegt die Chance, wieder Fahrt aufzunehmen. Und wir sehen wieder Licht auf der Strecke: Für dieses Jahr erwarten wir ein Wirtschaftswachstum von 1 Prozent und im nächsten Jahr von gut 1,3 Prozent. Das ist noch kein Wirtschaftsboom; aber es ist ein Anfang. Und die Signale sind ermutigend: Die Auftragseingänge im Inland steigen spürbar, vor allem bei den Investitionsgütern. Unsere Binnenwirtschaft wacht auf. Öffentliche Investitionen und der private Konsum gewinnen an Kraft. Das ist kein Zufall – das ist Ergebnis einer gezielten Wirtschafts- und Finanzpolitik. Wir haben mit den beiden Sondervermögen das größte Investitionsprogramm seit Jahrzehnten aufgelegt. Der Bund stellt allein in diesem Jahr 129 Milliarden Euro an Investitionen bereit, 20 Milliarden Euro mehr als 

In [354]:
for item in agenda_items:
    speeches = first_agenda_item.findall("rede")

    for speech in speeches:
        speech_paragraphs = speech.findall("p")

        speech_chunks =[]
        comments = []

        position = 0
        for element in speech:

            if element.tag == "p" and element.text:
                speech_chunks.append(element.text)
                position += (len(element.text) + 1)
            elif element.tag == "kommentar" and element.text:
                comments.append({"index_position": position,
                                 "comment_text": element.text})

        print(20*"+","SPEECH:", 20*"+")
        print(" ".join(speech_chunks))

        print("\n")
        print(20*"+","COMMENTS:", 20*"+")
        print(comments)
        print(100*"*")
        print(2*"\n")

++++++++++++++++++++ SPEECH: ++++++++++++++++++++
Frau Präsidentin! Meine sehr geehrten Damen und Herren! Hinter uns liegen zwei Jahre Rezession – Rückwärtsgang. Hinter uns liegt ein Jahr Stagnation – Seitwärtsgang. Vor uns liegt die Chance, wieder Fahrt aufzunehmen. Und wir sehen wieder Licht auf der Strecke: Für dieses Jahr erwarten wir ein Wirtschaftswachstum von 1 Prozent und im nächsten Jahr von gut 1,3 Prozent. Das ist noch kein Wirtschaftsboom; aber es ist ein Anfang. Und die Signale sind ermutigend: Die Auftragseingänge im Inland steigen spürbar, vor allem bei den Investitionsgütern. Unsere Binnenwirtschaft wacht auf. Öffentliche Investitionen und der private Konsum gewinnen an Kraft. Das ist kein Zufall – das ist Ergebnis einer gezielten Wirtschafts- und Finanzpolitik. Wir haben mit den beiden Sondervermögen das größte Investitionsprogramm seit Jahrzehnten aufgelegt. Der Bund stellt allein in diesem Jahr 129 Milliarden Euro an Investitionen bereit, 20 Milliarden Euro mehr als 

In [355]:
# Lets try to bring all that we discovered so far together to build a list of dictionaries
# that hold all information that we can extract from a session.
# Namely that means session level information, agenda level information, speaker level information
# and speech level information

# Lets quickly recap all info that we can and want attain at this point
## Session level data:
print(plenary_session_metadata)
issn_id = plenary_session_metadata["issn"]
plenary_period = plenary_session_metadata["wahlperiode"]
session_nr = plenary_session_metadata["sitzung-nr"]
session_date = plenary_session_metadata["sitzung-datum"]
session_start_time = plenary_session_metadata["sitzung-start-uhrzeit"]
session_end_time = plenary_session_metadata["sitzung-ende-uhrzeit"]
next_session_date = plenary_session_metadata["sitzung-naechste-datum"]


{'vertrieb': 'Bundesanzeiger Verlag GmbH, Postfach 1 0 05 34, 50445 Köln, Telefon (02 21) 97 66 83 40, Fax (02 21) 97 66 83 44, www.bundesanzeiger-verlag.de', 'herstellung': 'H. Heenemann GmbH  Co. KG, Buch- und Offsetdruckerei, Bessemerstraße 83–91, 12103 Berlin, www.heenemann-druck.de', 'sitzung-ort': 'Berlin', 'herausgeber': 'Deutscher Bundestag', 'issn': '0722-7980', 'wahlperiode': '21', 'sitzung-nr': '57', 'sitzung-datum': '30.01.2026', 'sitzung-start-uhrzeit': '09:00', 'sitzung-ende-uhrzeit': '15:00', 'sitzung-naechste-datum': '25.02.2026', 'start-seitennr': '6839'}


In [356]:
print(f"ISSN: {issn_id}")
print(f"Plenary Period:{plenary_period}")
print(f"Session Nr: {session_nr}")
print(f"Session Date: {session_date}")
print(f"Start Time: {session_start_time}")
print(f"End time: {session_end_time}")
print(f"Nex Session Date: {next_session_date}")

ISSN: 0722-7980
Plenary Period:21
Session Nr: 57
Session Date: 30.01.2026
Start Time: 09:00
End time: 15:00
Nex Session Date: 25.02.2026


In [357]:
# Agenda level info
agenda = root.find("sitzungsverlauf")
agenda_items = [item for item in agenda if item.tag != "sitzungsbeginn" and item.tag != "sitzungsende"]
nr_agenda_items = len(agenda_items)
agenda_item_names = [item.attrib.get("top-id", "No name found") for item in agenda_items]

In [358]:
print(f"Nr. of agenda items: {nr_agenda_items}")
print(f"Agenda item tags: {agenda_item_names}")

Nr. of agenda items: 7
Agenda item tags: ['Tagesordnungspunkt 7', 'Tagesordnungspunkt 24', 'Tagesordnungspunkt', 'Zusatzpunkt 8', 'Tagesordnungspunkt 5', 'Tagesordnungspunkt 26', 'Zusatzpunkt 9']


In [359]:
# Once we got the session level information we will loop over each agenda item
# , get its metadata (in our case its agenda_item_name, titles and doc relations)
# and then loop over its sub-elements namely speeches, which can be divided into speaker level information
# and speech contents

In [360]:
for item in agenda_items:
    agenda_name = item.get("top-id")
    agenda_titles = [title_element.text.strip() for title_element in item.findall(".//p[@klasse = 'T_fett']")]

    if len(agenda_titles) > 1:
        agenda_title = " / ".join(agenda_titles)
    elif agenda_titles:
        agenda_title = agenda_titles[0].strip()

    agenda_subtitles = [subtitle_element.text.strip() for subtitle_element in item.findall(".//p[@klasse = 'T_NaS']")]
    if len(agenda_subtitles) > 1:
        agenda_subtitle = " ".join(agenda_subtitles)
    elif agenda_subtitles:
        agenda_subtitle = agenda_subtitles[0].strip()

    agenda_doc_elements = item.findall(".//p[@klasse = 'T_Drs']/a")

    if agenda_doc_elements:
        agenda_doc_urls = [doc.get("href") for doc_element in agenda_doc_elements]

    print(f"{20*'-'} Agenda Name: {agenda_name} {20*'+'}")
    print(f"Agenda Title: {agenda_title}")
    print(f"Agenda Subtitle: {agenda_subtitle}")
    print(f"Agenda URLS: {agenda_doc_urls}")
    print(2*"\n")

-------------------- Agenda Name: Tagesordnungspunkt 7 ++++++++++++++++++++
Agenda Title: Zum Jahreswirtschaftsbericht 2026 / Jahreswirtschaftsbericht 2026 der Bundesregierung
Agenda Subtitle: 7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: b)	Beratung der Unterrichtung durch die Bundesregierung
Agenda URLS: ['https://dserver.bundestag.de/btd/21/028/2102804.pdf']



-------------------- Agenda Name: Tagesordnungspunkt 24 ++++++++++++++++++++
Agenda Title: Mobilitätsgarantie einführen – Produktionskapazitäten für die Verkehrswende aufbauen / Gemeindeverkehrsfinanzierungsgesetz novellieren – Kommunen stärken und Ausbau des öffentlichen Personennahverkehrs langfristig absichern
Agenda Subtitle: 24	Beratung des Antrags der Abgeordneten Luigi Pantisano, Marcel Bauer, Lorenz Gösta Beutin, weiterer Abgeordneter und der Fraktion Die Linke
Agenda URLS: ['https://dserver.bundestag.de/btd/21/028/2102804.pdf', 'https://dserver.bundestag.de/btd/21/028/21

In [361]:
speeches = first_agenda_item
for speech in speeches:
        speaker_id = speech.find(".//redner").get("id", None)
        name = speech.find(".//vorname")
        lastname = speech.find(".//nachname")
        role = speech.find(".//rolle_lang")
        party_affiliation = speech.find(".//fraktion")
        speakers_extracted.add(speaker_id)

        print(f"Speaker ID: {speaker_id}")
        print(f"Firstname: {name.text}")
        print(f"Lastname: {lastname.text}")
        if party_affiliation is not None:
            print(f"Party:: {party_affiliation.text}")
        if role is not None:
            print(f"Role: {role.text}")
        print(50* "+")

AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
for item in agenda_items:
    speeches = item.findall("rede")


    for speech in speeches:
        # get speaker information
        speaker_id = speech.find(".//redner").get("id", None)
        name = speech.find(".//vorname")
        lastname = speech.find(".//nachname")
        role = speech.find(".//rolle_lang")
        party_affiliation = speech.find(".//fraktion")
        speakers_extracted.add(speaker_id)

        # get speech information
        speech_paragraphs = speech.findall("p")

        speech_chunks =[]
        comments = []

        position = 1
        for element in speech:

            if element.tag == "p" and element.text:
                speech_chunks.append(element.text)
                position += (len(element.text) + 1)
            elif element.tag == "kommentar" and element.text:
                comments.append({"index_position": position,
                                 "comment_text": element.text})

In [ ]:
def get_text(element, xpath:str):
    """Safely extract text from an xml element"""
    found = element.find(xpath)

    return found.text.strip() if found is not None and found.text else None

In [ ]:
def get_agenda_info(agenda_item) -> dict:
    agenda_name = agenda_item.get("top-id")
    agenda_titles = [title_element.text.strip() for title_element in agenda_item.findall(".//p[@klasse = 'T_fett']")]

    if len(agenda_titles) > 1:
        agenda_title = " / ".join(agenda_titles)
    elif agenda_titles:
        agenda_title = agenda_titles[0].strip()

    agenda_subtitles = [subtitle_element.text.strip() for subtitle_element in agenda_item.findall(".//p[@klasse = 'T_NaS']")]
    if len(agenda_subtitles) > 1:
        agenda_subtitle = " ".join(agenda_subtitles)
    elif agenda_subtitles:
        agenda_subtitle = agenda_subtitles[0].strip()

    agenda_doc_elements = agenda_item.findall(".//p[@klasse = 'T_Drs']/a")

    if agenda_doc_elements:
        agenda_doc_urls = [doc.get("href") for doc_element in agenda_doc_elements]

    return {
            "angenda_name": agenda_name,
            "agenda_title": agenda_title,
            "agenda_subtitle": agenda_subtitle,
            "agenda_docs": agenda_doc_urls
            }

In [ ]:
def get_speaker_info(speech_raw) -> dict:
    """
    Expects a speech as an xml element, extracts all speaker attributes and returns them as a dictionary.
    :param speech_raw: xml speech element
    :return: dictionary with extracted speaker attributes
    """
    redner = speech_raw.find(".//redner")
    speaker_id = redner.get("id") if redner is not None else None

    return {
            "speaker_id" : speaker_id,
            "name" : get_text(speech_raw, ".//vorname"),
            "lastname" : get_text(speech_raw,".//nachname"),
            "role" : get_text(speech_raw,".//rolle_lang"),
            "party_affiliation" : get_text(speech_raw,".//fraktion"),
            }


In [ ]:
speaker_test = get_speaker_info(speech_raw=speech)
print(speaker_test)

In [ ]:
def get_speech_content(speech_raw)-> dict:
    """
    :param speech_raw: xml speech element
    :return: dictionary with whole speech as one string, and comments as a dict with index positions within speeech
    """
    speech_chunks =[]
    comments = []

    speech_id = speech_raw.get("id")

    position = 1
    for element in speech_raw:

        if element.tag == "p" and element.text:
            speech_chunks.append(element.text)
            position += (len(element.text) + 1)
        elif element.tag == "kommentar" and element.text:
            comments.append({"index_position": position,
                                "comment_text": element.text})
    speech_string = " ".join(speech_chunks)

    return {"speech_id": speech_id, "speech": speech_string, "comments": comments}

In [ ]:
test_speech = get_speech_content(speech)
print(test_speech)

In [ ]:
test_dict = {**get_speaker_info(speech), **get_speech_content(speech)}

In [ ]:
pprint(test_dict)

In [ ]:
def get_session_info(root)->dict:

    if root.tag =="dbtplenarprotokoll" and root.attrib:
        plenary_session_metadata = root.attrib

    return {"issn_id" : plenary_session_metadata["issn"],
            "plenary_period" : plenary_session_metadata["wahlperiode"],
            "session_nr" : plenary_session_metadata["sitzung-nr"],
            "session_date" : plenary_session_metadata["sitzung-datum"],
            "session_start_time" : plenary_session_metadata["sitzung-start-uhrzeit"],
            "session_end_time" : plenary_session_metadata["sitzung-ende-uhrzeit"],
            "next_session_date" : plenary_session_metadata["sitzung-naechste-datum"],
            }



In [ ]:
test_meta = get_session_info(root)
pprint(test_meta)

In [ ]:
agenda_test = get_agenda_info(first_agenda_item)
pprint(agenda_test)

In [368]:
df = pd.read_csv("./session.csv")

In [369]:
df.head()

,Unnamed: 0,issn_id,plenary_period,session_nr,session_date,session_start_time,session_end_time,next_session_date,agenda_name,agenda_title,agenda_subtitle,agenda_docs,speaker_id,name,lastname,role,party_affiliation,speech_id,speech,comments
0,0,0722-7980,21,24,17.09.2025,09:00,21:37,18.09.2025,Einzelplan 04,NaN,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11004930,Alice,Weidel,NaN,AfD,ID212400100,Sehr geehrte Frau Präsidentin! Sehr geehrter H...,"[{'index_position': 511, 'comment_text': '(Bei..."
1,1,0722-7980,21,24,17.09.2025,09:00,21:37,18.09.2025,Einzelplan 04,NaN,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11002735,Friedrich,Merz,Bundeskanzler,NaN,ID212400200,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 452, 'comment_text': '(Ste..."
2,2,0722-7980,21,24,17.09.2025,09:00,21:37,18.09.2025,Einzelplan 04,NaN,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11004263,Katharina,Dröge,NaN,BÜNDNIS 90/DIE GRÜNEN,ID212400300,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 131, 'comment_text': '(Jen..."
3,3,0722-7980,21,24,17.09.2025,09:00,21:37,18.09.2025,Einzelplan 04,NaN,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11003809,Matthias,Miersch,NaN,SPD,ID212400400,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,"[{'index_position': 831, 'comment_text': '(Tin..."
4,4,0722-7980,21,24,17.09.2025,09:00,21:37,18.09.2025,Einzelplan 04,NaN,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11005186,Heidi,Reichinnek,NaN,Die Linke,ID212400500,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 709, 'comment_text': '(Bei..."


In [370]:
print(len(df.columns))
df.columns

20


Index(['Unnamed: 0', 'issn_id', 'plenary_period', 'session_nr', 'session_date',
       'session_start_time', 'session_end_time', 'next_session_date',
       'agenda_name', 'agenda_title', 'agenda_subtitle', 'agenda_docs',
       'speaker_id', 'name', 'lastname', 'role', 'party_affiliation',
       'speech_id', 'speech', 'comments'],
      dtype='str')

In [371]:
df.iloc[:,10:]

,agenda_subtitle,agenda_docs,speaker_id,name,lastname,role,party_affiliation,speech_id,speech,comments
0,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11004930,Alice,Weidel,NaN,AfD,ID212400100,Sehr geehrte Frau Präsidentin! Sehr geehrter H...,"[{'index_position': 511, 'comment_text': '(Bei..."
1,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11002735,Friedrich,Merz,Bundeskanzler,NaN,ID212400200,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 452, 'comment_text': '(Ste..."
2,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11004263,Katharina,Dröge,NaN,BÜNDNIS 90/DIE GRÜNEN,ID212400300,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 131, 'comment_text': '(Jen..."
3,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11003809,Matthias,Miersch,NaN,SPD,ID212400400,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,"[{'index_position': 831, 'comment_text': '(Tin..."
4,a)\thier: Einzelplan 04 Bundeskanzler und Bund...,['https://dserver.bundestag.de/btd/21/010/2101...,11005186,Heidi,Reichinnek,NaN,Die Linke,ID212400500,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 709, 'comment_text': '(Bei..."
...,...,...,...,...,...,...,...,...,...,...
142,hier: Einzelplan 15 Bundesministerium für Gesu...,['https://dserver.bundestag.de/btd/21/010/2101...,11005174,Christos,Pantazis,NaN,SPD,ID212414300,"Frau Präsidentin! Frau Baum, noch einmal: Ich ...","[{'index_position': 105, 'comment_text': '(Bei..."
143,hier: Einzelplan 15 Bundesministerium für Gesu...,['https://dserver.bundestag.de/btd/21/010/2101...,11005248,Johannes,Wagner,NaN,BÜNDNIS 90/DIE GRÜNEN,ID212414400,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,"[{'index_position': 224, 'comment_text': '(Bei..."
144,hier: Einzelplan 15 Bundesministerium für Gesu...,['https://dserver.bundestag.de/btd/21/010/2101...,11004829,Axel,Müller,NaN,CDU/CSU,ID212414500,Sehr geehrte Frau Präsidentin! Werte Kolleginn...,"[{'index_position': 646, 'comment_text': '(Bei..."
145,hier: Einzelplan 15 Bundesministerium für Gesu...,['https://dserver.bundestag.de/btd/21/010/2101...,11005419,Joachim,Bloch,NaN,AfD,ID212414600,Frau Präsidentin! Frau Ministerin! Liebe Kolle...,"[{'index_position': 785, 'comment_text': '(Bei..."


In [372]:
df["party_affiliation"].value_counts().sort_values(ascending=False)

party_affiliation
AfD                      38
CDU/CSU                  33
SPD                      26
BÜNDNIS 90/DIE GRÜNEN    22
Die Linke                18
fraktionslos              1
SPDSPD                    1
Name: count, dtype: int64

In [373]:
df["agenda_name"].value_counts().sort_values(ascending=False)

agenda_name
Einzelplan 04    34
Einzelplan 15    23
Einzelplan 23    21
Einzelplan 05    18
Einzelplan 11    18
Einzelplan 30    17
Einzelplan 14    16
Name: count, dtype: int64

In [374]:
df.dtypes

Unnamed: 0              int64
issn_id                   str
plenary_period          int64
session_nr              int64
session_date              str
session_start_time        str
session_end_time          str
next_session_date         str
agenda_name               str
agenda_title          float64
agenda_subtitle           str
agenda_docs               str
speaker_id                str
name                      str
lastname                  str
role                      str
party_affiliation         str
speech_id                 str
speech                    str
comments                  str
dtype: object